<a href="https://colab.research.google.com/github/mitalidaduria/nlp-payments-lab/blob/main/SHAP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q shap xgboost scikit-learn pandas numpy matplotlib

In [2]:
import logging
import numpy as np
import pandas as pd
import shap

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


class FraudExplainer:
    """SHAP-based explainability module for tree-based fraud detection models.

    Provides local prediction explanations, regulatory mapping, and data
    leakage detection.
    """

    # Human-readable translations for technical feature names (GDPR Article 22 Requirement)
    FEATURE_TRANSLATIONS = {
        "is_night": "unusual transaction timing",
        "is_high_velocity": "high recent transaction frequency",
        "amount_usd": "unusually high transaction amount",
        "distance_from_home": "transaction location far from primary residence",
        "failed_pin_attempts": "multiple consecutive failed PIN attempts",
        "foreign_transaction": "transaction originated overseas",
    }

    def __init__(self, model, feature_names: list[str]):
        """Initialize the SHAP TreeExplainer.

        :param model: Trained tree model (e.g., XGBoost, LightGBM,
        RandomForest)
        :param feature_names: List of feature names used during training
        """
        self.model = model
        self.feature_names = feature_names
        # TreeExplainer is fast and exact for tree-based architectures
        self.explainer = shap.TreeExplainer(model)

    def explain_prediction(self, sample: pd.DataFrame, top_k: int = 3) -> dict:
        """Generates local SHAP explanations for a single transaction sample.

        :param sample: Single-row pandas DataFrame containing transaction
        features
        :param top_k: Number of top driving features to return in each direction
        :return: Dict containing base value, top risk drivers, and top
        fraud-reducing factors
        """
        shap_values = self.explainer(sample)

        # Handle 1D array extraction for a single transaction sample
        sample_shap = shap_values.values[0]
        base_value = shap_values.base_values[0]

        feature_shap_df = pd.DataFrame({
            "feature": self.feature_names,
            "value": sample.values[0],
            "shap_value": sample_shap,
        })

        # Features pushing the prediction higher (increasing risk)
        risk_drivers = (
            feature_shap_df[feature_shap_df["shap_value"] > 0]
            .sort_values(by="shap_value", ascending=False)
            .head(top_k)
            .to_dict(orient="records")
        )

        # Features pushing the prediction lower (reducing risk)
        risk_reducers = (
            feature_shap_df[feature_shap_df["shap_value"] < 0]
            .sort_values(by="shap_value")
            .head(top_k)
            .to_dict(orient="records")
        )

        return {
            "base_value": float(base_value),
            "prediction_score": float(base_value + sample_shap.sum()),
            "risk_drivers": risk_drivers,
            "risk_reducers": risk_reducers,
        }

    def regulatory_explanation(
        self, sample: pd.DataFrame, top_k: int = 2
    ) -> str:
        """GDPR Article 22 compliant natural language explanation for a decision.

        :param sample: Single-row pandas DataFrame containing transaction
        features
        :param top_k: Number of key factors to present to the user
        :return: Human-readable sentence explaining why the transaction was
        flagged/declined
        """
        explanation = self.explain_prediction(sample, top_k=top_k)
        risk_drivers = explanation["risk_drivers"]

        if not risk_drivers:
            return "This transaction was approved based on normal account activity."

        reasons = []
        for driver in risk_drivers:
            feature_name = driver["feature"]
            # Fallback to feature name if explicit translation mapping is missing
            human_label = self.FEATURE_TRANSLATIONS.get(
                feature_name, feature_name.replace("_", " ")
            )
            reasons.append(human_label)

        reason_text = " and ".join(reasons)
        return f"This transaction was flagged for security review based on: {reason_text}."

    def detect_data_leakage(
        self,
        X_train: pd.DataFrame,
        suspicious_cols: list[str],
        threshold: float = 0.15,
    ) -> bool:
        """Checks if identifier columns (like user_id) hold unusually high
        average SHAP importance.

        :param X_train: Training dataset sample
        :param suspicious_cols: List of ID or metadata columns to test (e.g.,
        ['user_id'])
        :param threshold: Fraction of total importance threshold to trigger
        leakage alert
        :return: True if potential leakage is detected, False otherwise
        """
        print("\nCalculating SHAP global importance to audit data leakage...")
        shap_values = self.explainer.shap_values(X_train)

        # Calculate mean absolute SHAP value across all training instances
        mean_abs_shap = np.abs(shap_values).mean(axis=0)
        total_importance = mean_abs_shap.sum()

        importance_df = pd.DataFrame({
            "feature": self.feature_names,
            "importance": mean_abs_shap / total_importance,
        }).sort_values(by="importance", ascending=False)

        leakage_detected = False
        for col in suspicious_cols:
            if col in importance_df["feature"].values:
                col_importance = importance_df.loc[
                    importance_df["feature"] == col, "importance"
                ].values[0]

                if col_importance > threshold:
                    print(
                        f"🚨 DATA LEAKAGE DETECTED! '{col}' accounts for "
                        f"{col_importance:.2%} of total model decisions. "
                        f"The model is memorizing IDs instead of learning general patterns."
                    )
                    leakage_detected = True

        if not leakage_detected:
            print(
                "✅ Data leakage check passed. No target/ID leakage detected."
            )

        return leakage_detected

In [3]:
import pandas as pd
import xgboost as xgb

# 1. Synthesize sample fraud dataset
data = pd.DataFrame({
    "amount_usd": [1500.0, 20.0, 3200.0, 15.0],
    "is_night": [1, 0, 1, 0],
    "is_high_velocity": [1, 0, 1, 0],
    "failed_pin_attempts": [2, 0, 3, 0],
    "user_id": [101, 102, 103, 104],  # Metadata column to evaluate leakage
})
y = [1, 0, 1, 0]  # Target: 1 = Fraud, 0 = Legitimate

feature_cols = [
    "amount_usd",
    "is_night",
    "is_high_velocity",
    "failed_pin_attempts",
    "user_id",
]
X = data[feature_cols]

# 2. Fit XGBoost classifier
model = xgb.XGBClassifier(n_estimators=10, max_depth=3, random_state=42)
model.fit(X, y)

# 3. Instantiate FraudExplainer
explainer = FraudExplainer(model=model, feature_names=feature_cols)

# 4. Generate GDPR Article 22 human explanation for Row 0 (High Risk)
sample_transaction = X.iloc[[0]]
print("--- 1. GDPR Article 22 Natural Language Explanation ---")
print(explainer.regulatory_explanation(sample_transaction, top_k=2))

# 5. Inspect SHAP details
print("\n--- 2. Raw SHAP Output Breakdown ---")
details = explainer.explain_prediction(sample_transaction)
print(f"Base Value: {details['base_value']:.4f}")
print("Top Risk Drivers:", details["risk_drivers"])

# 6. Run Data Leakage Audit on user_id
print("\n--- 3. Data Leakage Audit ---")
explainer.detect_data_leakage(
    X_train=X, suspicious_cols=["user_id"], threshold=0.10
)

--- 1. GDPR Article 22 Natural Language Explanation ---
This transaction was approved based on normal account activity.

--- 2. Raw SHAP Output Breakdown ---
Base Value: 0.0000
Top Risk Drivers: []

--- 3. Data Leakage Audit ---

Calculating SHAP global importance to audit data leakage...
✅ Data leakage check passed. No target/ID leakage detected.


/tmp/ipykernel_1579/103746672.py:137: RuntimeWarning: invalid value encountered in divide
  "importance": mean_abs_shap / total_importance,


False